In [1]:
import pandas as pd
import numpy as np

In [2]:
df1 = pd.read_csv(r"C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\KEN_2012-2023_RTC_v01_M_CSV\ma3route_crashes_algorithmcode.csv")
df2 = pd.read_csv(r"C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\KEN_2012-2023_RTC_v01_M_CSV\ma3route_crashes_manualcode.csv")

df = pd.concat([df1, df2], ignore_index=True)

In [3]:
df.sample(5)

,crash_id,crash_datetime,crash_date,latitude,longitude,n_crash_reports,contains_fatality_words,contains_pedestrian_words,contains_matatu_words,contains_motorcycle_words
28727,28728,2015-06-30 10:28:49,2015-06-30,-1.334437,36.883188,2,0,0,0,0
33097,2380,2018-06-12 08:01:24,2018-06-12,-1.195495,36.924687,2,0,0,1,0
6680,6681,2021-10-31 19:00:48,2021-10-31,-1.231189,36.670164,1,0,0,0,0
33092,2375,2018-06-07 21:49:54,2018-06-07,-1.316698,36.833959,2,0,0,0,0
10713,10714,2014-01-21 09:47:34,2014-01-21,-1.266624,36.746475,1,0,0,0,0


In [4]:
%pip install earthengine-api


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install --no-deps geemap
%pip install ipyleaflet folium bqplot ipyfilechooser ipyevents ee-extra pyshp pycrs scooby colour


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install eerepr box-python-sdk gdown sankee


  Using cached eerepr-0.1.4-py3-none-any.whl.metadata (4.2 kB)
  Using cached box_python_sdk-0.0.3-py2.py3-none-any.whl.metadata (1.2 kB)
  Using cached gdown-6.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached sankee-0.3.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached filelock-4.2.10-py3-none-any.whl.metadata (2.3 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached plotly-5.24.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached soupsieve-4.1.1-py3-none-any.whl.metadata (4.4 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached eerepr-0.1.4-py3-none-any.whl (9.6 kB)
Using cached box_python_sdk-0.0.3-py2.py3-none-any.whl (5.5 kB)
Using cached gdown-6.5.1-py3-none-any.whl (51 kB)
Using cached sankee-0.3.0-py3-none-any.whl (20 kB)
   ---------------------------------------- 0.0/19.1 MB ? eta -:

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
geemap 0.39.0 requires anywidget, which is not installed.
geemap 0.39.0 requires geocoder, which is not installed.
geemap 0.39.0 requires pyperclip, which is not installed.
geemap 0.39.0 requires python-box, which is not installed.
geemap 0.39.0 requires xee>=0.1.0, which is not installed.

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
%pip install --no-deps python-box


  Using cached python_box-8.0.1-cp312-cp312-win_amd64.whl.metadata (10 kB)
Using cached python_box-8.0.1-cp312-cp312-win_amd64.whl (1.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
%pip install --no-deps anywidget psygnal


  Using cached anywidget-0.11.0-py3-none-any.whl.metadata (8.9 kB)
  Using cached psygnal-0.16.1-cp312-cp312-win_amd64.whl.metadata (6.5 kB)
Using cached anywidget-0.11.0-py3-none-any.whl (317 kB)
Using cached psygnal-0.16.1-cp312-cp312-win_amd64.whl (424 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import ee, geemap

ee.Initialize(project='numeric-anthem-514816-m2')

# Same extent as the KMD dataset: 33.8E–42E, 4.8S–5.2N
aoi = ee.Geometry.Rectangle([33.8, -4.8, 42.0, 5.2])

chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterBounds(aoi)
          .filterDate('2012-07-01', '2018-12-31')
          .select('precipitation'))

print('Number of daily images:', chirps.size().getInfo())   # expect 2375

# Reduce the collection to a single image for display
mean_precip = chirps.mean().clip(aoi)

vis = {
    'min': 0,
    'max': 8,        # mm/day — tune this; the 1–17 range in your snippet suits storm days, not means
    'palette': ['001137', '0aab1e', 'e7eb05', 'ff4a2d', 'e90000'],
}

m = geemap.Map()
m.set_center(37.9, 0.2, 6)      # centred on Kenya
m.add_basemap('HYBRID')
m.add_layer(mean_precip, vis, 'CHIRPS mean daily rainfall 2012–2018')
m

EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

In [20]:
import ee, geemap

ee.Initialize(project='numeric-anthem-514816-m2')

# Same extent as the KMD dataset: 33.8E–42E, 4.8S–5.2N
aoi = ee.Geometry.Rectangle([33.8, -4.8, 42.0, 5.2])
print("Success!")


Success!


In [21]:
chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterBounds(aoi)
          .filterDate('2012-07-01', '2018-12-31')
          .select('precipitation'))

print('Number of daily images:', chirps.size().getInfo())   # expect 2375

# Reduce the collection to a single image for display
mean_precip = chirps.mean().clip(aoi)

vis = {
    'min': 0,
    'max': 8,        # mm/day — tune this; the 1–17 range in your snippet suits storm days, not means
    'palette': ['001137', '0aab1e', 'e7eb05', 'ff4a2d', 'e90000'],
}

m = geemap.Map()
m.set_center(37.9, 0.2, 6)      # centred on Kenya
m.add_basemap('HYBRID')
m.add_layer(mean_precip, vis, 'CHIRPS mean daily rainfall 2012–2018')
m

Number of daily images: 2374


Map(center=[0.2, 37.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

In [25]:
# CHIRPS collection over the FULL crash period
chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterBounds(aoi)
          .filterDate('2012-07-01', '2023-08-01')   # exclusive end -> covers through 31 Jul 2023
          .select('precipitation'))

print('Total daily images:', chirps.size().getInfo())   # ~4048

for y in range(2012, 2024):
    start = '2012-07-01' if y == 2012 else f'{y}-01-01'
    end   = '2023-08-01' if y == 2023 else f'{y+1}-01-01'

    col = chirps.filterDate(start, end)
    print(y, col.size().getInfo(), 'days')

    ee.batch.Export.image.toDrive(
        image=col.toBands(),                      # 1 band per day, named e.g. 20180101_precipitation
        description=f'chirps_kenya_daily_{y}',
        folder='GEE_Kenya',
        region=aoi,
        scale=5566,                               # CHIRPS native resolution
        crs='EPSG:4326',
        maxPixels=1e10,
    ).start()
    

Total daily images: 4048
2012 184 days
2013 365 days
2014 365 days
2015 365 days
2016 366 days
2017 365 days
2018 365 days
2019 365 days
2020 366 days
2021 365 days
2022 365 days
2023 212 days


In [28]:
print(aoi.getInfo()['type'])
print(aoi.bounds().getInfo()['coordinates'])


Polygon
[[[33.8, -4.812257996556922], [42, -4.812257996556922], [42, 5.213268650247928], [33.8, 5.213268650247928], [33.8, -4.812257996556922]]]


In [30]:
%pip install rasterio


  Using cached affine-3.0.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached click-8.5.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import os, glob

base = r"C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001"

# Recursive: find every .tif anywhere under that folder
files = sorted(glob.glob(os.path.join(base, "**", "*.tif"), recursive=True))

print(len(files), "tif files found")
for f in files:
    print(" ", f)

12 tif files found
  C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001\chirps_kenya_daily_2012.tif
  C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001\chirps_kenya_daily_2013.tif
  C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001\chirps_kenya_daily_2014.tif
  C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001\chirps_kenya_daily_2015.tif
  C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001\chirps_kenya_daily_2016.tif
  C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001\chirps_kenya_daily_2017.tif
  C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001\chirps_kenya_daily_2018.tif
  C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001\chirps_kenya_daily_2019.tif
  C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DA

In [37]:
import os, glob, rasterio
import numpy as np
import pandas as pd

base = r"C:\Users\HomePC\ROAD-ACCIDENT-IN-NAIROBI-DATASET\drive-download-20261115T182053Z-1-001"
files = sorted(glob.glob(os.path.join(base, "**", "*.tif"), recursive=True))
print(len(files), "year files found\n")

crashes = df.copy()
crashes['date_str'] = pd.to_datetime(crashes['crash_date']).dt.strftime('%Y%m%d')
crashes['precipitation'] = np.nan

for fp in files:
    name = os.path.basename(fp)            # <- Windows-safe
    with rasterio.open(fp) as ds:
        band_of = {nm.split('_')[0]: i
                   for i, nm in enumerate(ds.descriptions, start=1)
                   if nm}

        sub = crashes[crashes['date_str'].isin(band_of.keys())]
        if sub.empty:
            print(f'{name}: 0 crash dates matched, skipping')
            continue

        matched = 0
        for date_str, grp in sub.groupby('date_str'):
            idx = band_of[date_str]
            vals = [v[0] for v in ds.sample(
                list(zip(grp['longitude'], grp['latitude'])),
                indexes=idx
            )]
            crashes.loc[grp.index, 'precipitation'] = vals
            matched += len(grp)

        print(f'{name}: {matched} crashes filled')

print('\n--- summary ---')
print('NaN precipitation rows :', crashes['precipitation'].isna().sum())
print('Total crashes          :', len(crashes))
print('\nPrecipitation stats (mm/day):')
print(crashes['precipitation'].describe())
print('\nSample:')
print(crashes[['crash_id', 'crash_date', 'latitude', 'longitude',
               'precipitation']].head(10))

12 year files found

chirps_kenya_daily_2012.tif: 310 crashes filled
chirps_kenya_daily_2013.tif: 1398 crashes filled
chirps_kenya_daily_2014.tif: 4014 crashes filled
chirps_kenya_daily_2015.tif: 6101 crashes filled
chirps_kenya_daily_2016.tif: 4219 crashes filled
chirps_kenya_daily_2017.tif: 3777 crashes filled
chirps_kenya_daily_2018.tif: 4710 crashes filled
chirps_kenya_daily_2019.tif: 2739 crashes filled
chirps_kenya_daily_2020.tif: 1928 crashes filled
chirps_kenya_daily_2021.tif: 1937 crashes filled
chirps_kenya_daily_2022.tif: 1805 crashes filled
chirps_kenya_daily_2023.tif: 721 crashes filled

--- summary ---
NaN precipitation rows : 4
Total crashes          : 33659

Precipitation stats (mm/day):
count    33655.000000
mean         2.842350
std          9.650788
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.302170
Name: precipitation, dtype: float64

Sample:
   crash_id  crash_date  latitude  longitude  precipitation
0     

In [38]:
crashes.to_csv('crashes_with_rainfall.csv', index=False)
print('saved', len(crashes), 'rows')

saved 33659 rows


In [39]:
crashes.head()

,crash_id,crash_datetime,crash_date,latitude,longitude,n_crash_reports,contains_fatality_words,contains_pedestrian_words,contains_matatu_words,contains_motorcycle_words,date_str,precipitation
0,1,2018-06-06 20:39:54,2018-06-06,-1.263030,36.764374,1,0,0,0,0,20180606,9.996126
1,2,2018-08-17 06:15:54,2018-08-17,-0.829710,37.037820,1,1,0,0,0,20180817,0.000000
2,3,2018-05-25 17:51:54,2018-05-25,-1.125301,37.003297,1,0,0,0,0,20180525,0.000000
3,4,2018-05-25 18:11:54,2018-05-25,-1.740958,37.129026,1,0,0,0,0,20180525,0.000000
4,5,2018-05-25 21:59:54,2018-05-25,-1.259392,36.842321,1,1,0,0,0,20180525,0.000000


In [40]:
bad = crashes[crashes['precipitation'].isna()]
print(bad[['crash_id', 'crash_datetime', 'crash_date', 'latitude', 'longitude']])

       crash_id       crash_datetime  crash_date  latitude  longitude
31450       387  2018-03-02 11:09:42  2018-03-02 -3.967212  39.751449
31512       449  2018-01-06 05:20:12  2018-01-06 -4.032846  39.650282
32510      1793  2017-11-23 07:00:10  2017-11-23 -2.863738  40.171389
32571      1854  2017-12-04 12:22:40  2017-12-04 -4.047225  39.695068


In [42]:
%pip install geopandas


  Using cached pyproj-3.8.0-cp312-cp312-win_amd64.whl.metadata (40 kB)
  Using cached shapely-2.2.0-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.8 MB ? eta -:--:--
    --------------------------------------- 0.5/23.8 MB 1.5 MB/s eta 0:00:16
   - -------------------------------------- 0.8/23.8 MB 1.2 MB/s eta 0:00:19
   - -------------------------------------- 1.0/23.8 MB 1.1 MB/s eta 0:00:21
   - -------------------------------------- 1.0/23.8 MB 1.1 MB/s eta 0:00:21
   -- ------------------------------------- 1.3/23.8 MB 1.0 MB/s eta 0:00:22
   -- ------------------------------------- 1.6/23.8 MB 1.0 MB/s eta 0:00:22
   --- ------------------------------------ 1.8/23.8 MB 1.0 MB/s eta 0:00:22
   --- ------------------------------------ 2.1/23.8 MB 1.0 MB/s eta 0:00:21
   --- -----------------------


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# CSV — keep the tabular version
crashes.to_csv('crashes_with_rainfall.csv', index=False)

# GeoPackage — spatial version you'll pick up in the next notebook
crashes_gdf = gpd.GeoDataFrame(
    crashes,
    geometry=gpd.points_from_xy(crashes['longitude'], crashes['latitude']),
    crs='EPSG:4326'
)
crashes_gdf.to_file('crashes_with_rainfall.gpkg', driver='GPKG', layer='crashes')

print('saved', len(crashes), 'crashes;',
      'spatial bounds:', crashes_gdf.total_bounds)

saved 33659 crashes; spatial bounds: [34.145808 -4.059868 40.171389  1.257331]
